In [1]:
import sys
print(sys.executable)

/home/vboxuser/mlops_assignment/.venv/bin/python3


In [2]:
import mlflow
import mlflow.sklearn
import pandas as pd
import numpy as np

from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score

mlflow.set_tracking_uri("http://localhost:5000")

# Separate experiment for this clean rerun
mlflow.set_experiment("mnist-mlp-rerun")

print("Tracking URI:", mlflow.get_tracking_uri())

/home/vboxuser/mlops_assignment/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Tracking URI: http://localhost:5000


In [3]:
mnist = fetch_openml("mnist_784", version=1, as_frame=False)

X = mnist.data.astype(np.float32) / 255.0
y = mnist.target.astype(np.int64)

# Keep the VM lightweight
X = X[:10000]
y = y[:10000]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)


def train_and_evaluate(
    learning_rate=0.001,
    batch_size=32,
    hidden_layer_sizes=(128,)
):
    model = MLPClassifier(
        hidden_layer_sizes=hidden_layer_sizes,
        learning_rate_init=learning_rate,
        batch_size=batch_size,
        max_iter=10,
        random_state=42,
        early_stopping=True,
        validation_fraction=0.1,
        n_iter_no_change=3
    )

    model.fit(X_train, y_train)

    train_preds = model.predict(X_train)
    test_preds = model.predict(X_test)

    train_accuracy = accuracy_score(y_train, train_preds)
    val_accuracy = accuracy_score(y_test, test_preds)

    train_loss = model.loss_

    return model, train_loss, train_accuracy, val_accuracy


_, train_loss, train_accuracy, val_accuracy = train_and_evaluate()

print(f"train_loss={train_loss:.4f}")
print(f"train_accuracy={train_accuracy:.4f}")
print(f"val_accuracy={val_accuracy:.4f}")

train_loss=0.0423
train_accuracy=0.9905
val_accuracy=0.9420


/home/vboxuser/mlops_assignment/.venv/lib/python3.14/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (10) reached and the optimization hasn't converged yet.
  warnings.warn(


In [4]:
def train_and_log(
    learning_rate=0.001,
    batch_size=32,
    hidden_layer_sizes=(128,),
    run_name=None
):
    with mlflow.start_run(run_name=run_name):

        mlflow.log_param("learning_rate", learning_rate)
        mlflow.log_param("batch_size", batch_size)
        mlflow.log_param("hidden_layer_sizes", hidden_layer_sizes)
        mlflow.log_param("max_iter", 10)

        model, train_loss, train_accuracy, val_accuracy = train_and_evaluate(
            learning_rate=learning_rate,
            batch_size=batch_size,
            hidden_layer_sizes=hidden_layer_sizes
        )

        mlflow.log_metric("train_loss", train_loss)
        mlflow.log_metric("train_accuracy", train_accuracy)
        mlflow.log_metric("val_accuracy", val_accuracy)

        mlflow.set_tag("model", "MLP")
        mlflow.set_tag("dataset", "MNIST")

        run_id = mlflow.active_run().info.run_id

        print(
            f"Logged run {run_id} | "
            f"lr={learning_rate} | "
            f"batch={batch_size} | "
            f"train_loss={train_loss:.4f} | "
            f"val_accuracy={val_accuracy:.4f}"
        )

        return run_id

In [5]:
sweep_run_ids = []

experiments = [
    (0.001, 32),
    (0.001, 64),
    (0.001, 128),
    (0.01, 32),
    (0.01, 64),
    (0.01, 128),
]

for learning_rate, batch_size in experiments:

    run_id = train_and_log(
        learning_rate=learning_rate,
        batch_size=batch_size,
        hidden_layer_sizes=(128,),
        run_name=f"mlp-lr-{learning_rate}-batch-{batch_size}"
    )

    sweep_run_ids.append(run_id)

print("\nSix experiment run IDs:")
for run_id in sweep_run_ids:
    print(run_id)

/home/vboxuser/mlops_assignment/.venv/lib/python3.14/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (10) reached and the optimization hasn't converged yet.
  warnings.warn(


Logged run 2e8507e0da424590bb0103d010295764 | lr=0.001 | batch=32 | train_loss=0.0423 | val_accuracy=0.9420
🏃 View run mlp-lr-0.001-batch-32 at: http://localhost:5000/#/experiments/3/runs/2e8507e0da424590bb0103d010295764
🧪 View experiment at: http://localhost:5000/#/experiments/3


/home/vboxuser/mlops_assignment/.venv/lib/python3.14/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (10) reached and the optimization hasn't converged yet.
  warnings.warn(


Logged run 1b1b532fc8ba4e0e820e9204bf25cc40 | lr=0.001 | batch=64 | train_loss=0.0738 | val_accuracy=0.9350
🏃 View run mlp-lr-0.001-batch-64 at: http://localhost:5000/#/experiments/3/runs/1b1b532fc8ba4e0e820e9204bf25cc40
🧪 View experiment at: http://localhost:5000/#/experiments/3


/home/vboxuser/mlops_assignment/.venv/lib/python3.14/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (10) reached and the optimization hasn't converged yet.
  warnings.warn(


Logged run 912eb5b2e35744dd875cf368d7e3657a | lr=0.001 | batch=128 | train_loss=0.1136 | val_accuracy=0.9330
🏃 View run mlp-lr-0.001-batch-128 at: http://localhost:5000/#/experiments/3/runs/912eb5b2e35744dd875cf368d7e3657a
🧪 View experiment at: http://localhost:5000/#/experiments/3
Logged run fd0b3abe891341f6ac8481b33c1d3bcf | lr=0.01 | batch=32 | train_loss=0.1203 | val_accuracy=0.9315
🏃 View run mlp-lr-0.01-batch-32 at: http://localhost:5000/#/experiments/3/runs/fd0b3abe891341f6ac8481b33c1d3bcf
🧪 View experiment at: http://localhost:5000/#/experiments/3


/home/vboxuser/mlops_assignment/.venv/lib/python3.14/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (10) reached and the optimization hasn't converged yet.
  warnings.warn(


Logged run e80c92e84ac64bd8965bee850130c813 | lr=0.01 | batch=64 | train_loss=0.0542 | val_accuracy=0.9475
🏃 View run mlp-lr-0.01-batch-64 at: http://localhost:5000/#/experiments/3/runs/e80c92e84ac64bd8965bee850130c813
🧪 View experiment at: http://localhost:5000/#/experiments/3


/home/vboxuser/mlops_assignment/.venv/lib/python3.14/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (10) reached and the optimization hasn't converged yet.
  warnings.warn(


Logged run 17a29fd30ad14a378996311d5ce43382 | lr=0.01 | batch=128 | train_loss=0.0143 | val_accuracy=0.9490
🏃 View run mlp-lr-0.01-batch-128 at: http://localhost:5000/#/experiments/3/runs/17a29fd30ad14a378996311d5ce43382
🧪 View experiment at: http://localhost:5000/#/experiments/3

Six experiment run IDs:
2e8507e0da424590bb0103d010295764
1b1b532fc8ba4e0e820e9204bf25cc40
912eb5b2e35744dd875cf368d7e3657a
fd0b3abe891341f6ac8481b33c1d3bcf
e80c92e84ac64bd8965bee850130c813
17a29fd30ad14a378996311d5ce43382


In [6]:
runs_df = mlflow.search_runs(
    experiment_names=["mnist-mlp-rerun"]
)

comparison = runs_df[
    [
        "tags.mlflow.runName",
        "params.learning_rate",
        "params.batch_size",
        "metrics.train_loss",
        "metrics.train_accuracy",
        "metrics.val_accuracy",
    ]
].copy()

comparison.columns = [
    "Run",
    "Learning Rate",
    "Batch Size",
    "Train Loss",
    "Train Accuracy",
    "Validation Accuracy",
]

for col in [
    "Learning Rate",
    "Batch Size",
    "Train Loss",
    "Train Accuracy",
    "Validation Accuracy"
]:
    comparison[col] = pd.to_numeric(
        comparison[col],
        errors="coerce"
    )

comparison = comparison.sort_values(
    ["Learning Rate", "Batch Size"]
).reset_index(drop=True)

comparison["Learning Rate"] = comparison["Learning Rate"].round(3)
comparison["Train Loss"] = comparison["Train Loss"].round(4)
comparison["Train Accuracy"] = comparison["Train Accuracy"].round(4)
comparison["Validation Accuracy"] = comparison["Validation Accuracy"].round(4)

display(
    comparison.style.hide(axis="index")
)

Run,Learning Rate,Batch Size,Train Loss,Train Accuracy,Validation Accuracy
mlp-lr-0.001-batch-32,0.001000,32,0.042300,0.990500,0.942000
mlp-lr-0.001-batch-32,0.001000,32,0.042300,0.990500,0.942000
mlp-lr-0.001-batch-64,0.001000,64,0.073800,0.983900,0.935000
mlp-lr-0.001-batch-64,0.001000,64,0.073800,0.983900,0.935000
mlp-lr-0.001-batch-128,0.001000,128,0.113600,0.974100,0.933000
mlp-lr-0.001-batch-128,0.001000,128,0.113600,0.974100,0.933000
mlp-lr-0.01-batch-32,0.010000,32,0.120300,0.973000,0.931500
mlp-lr-0.01-batch-32,0.010000,32,0.120300,0.973000,0.931500
mlp-lr-0.01-batch-64,0.010000,64,0.054200,0.993000,0.947500
mlp-lr-0.01-batch-64,0.010000,64,0.054200,0.993000,0.947500


In [7]:
best_run = comparison.loc[
    comparison["Validation Accuracy"].idxmax()
]

print("========== BEST RUN ==========")
print(f"Run                 : {best_run['Run']}")
print(f"Learning Rate       : {best_run['Learning Rate']:.3f}")
print(f"Batch Size          : {int(best_run['Batch Size'])}")
print(f"Train Loss          : {best_run['Train Loss']:.4f}")
print(f"Train Accuracy      : {best_run['Train Accuracy']:.4f}")
print(f"Validation Accuracy : {best_run['Validation Accuracy']:.4f}")
print("==============================")

========== BEST RUN ==========
Run                 : mlp-lr-0.01-batch-128
Learning Rate       : 0.010
Batch Size          : 128
Train Loss          : 0.0143
Train Accuracy      : 0.9920
Validation Accuracy : 0.9490
